In [22]:
import os
import json
import glob
import numpy as np
import xarray as xr
import xskillscore as xs
import matplotlib.pyplot as plt
from typing import Dict
from joblib import Parallel, delayed
from cftime import num2date
from datetime import timedelta

In [23]:
def get_reference_data(freq="monthly", data_root=None):
    # Validate frequency input
    valid_freqs = {"monthly", "daily"}
    if freq not in valid_freqs:
        raise ValueError(f"[ERROR] Unsupported frequency: '{freq}'. Supported options: {valid_freqs}")

    # Base path
    base_root = data_root or "/pscratch/sd/z/zhan391/seacrogs_scratch/post_data"
    era5_path = os.path.join(base_root, "ERA5", freq)
    ceres_path = os.path.join(base_root, "CERES-OAFlux", freq)
    gpcp_path = os.path.join(base_root, "GPCP", freq)

    # Common ERA5 template
    era5_template = "ERA5_analysis_monthly_%(year).nc"
    ceres_template = "CERES-OAFlux_%(year).nc"
    gpcp_template = "GPCP_%(year).nc"

    ref_dict = {
        # ERA5 Variables
        "PRECST":   {"source": "ERA5", "alias": "PRECST",   "fscl": 8.64e7,  "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "QFLX":     {"source": "ERA5", "alias": "MEP",      "fscl": 86400.0, "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "LHFLX":    {"source": "ERA5", "alias": "LHFLX",    "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "SHFLX":    {"source": "ERA5", "alias": "SHFLX",    "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "TGCLDLWP": {"source": "ERA5", "alias": "TGCLDLWP", "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "TGCLDIWP": {"source": "ERA5", "alias": "TGCLDIWP", "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "TREFHT":   {"source": "ERA5", "alias": "TREFHT",   "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "TMQ":      {"source": "ERA5", "alias": "TMQ",      "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "TS":       {"source": "ERA5", "alias": "TS",       "fscl": 1.0,     "run": era5_path, "template": era5_template, "period": "197901_201812"},
        "PSL":      {"source": "ERA5", "alias": "PSL",      "fscl": 1e-2,    "run": era5_path, "template": era5_template, "period": "197901_201812"},

        # CERES/OAFlux Variables
        **{name: {"source": "CERES", "alias": alias, "fscl": 1.0, "run": ceres_path, "template": ceres_template, "period": "200101-201812"}
           for name, alias in {
                "FLUT": "OLR", "FLUTC": "OLR", "FLDS": "FLDS", "FLDSC": "FLDSC",
                "FSNS": "FSNS", "FLNS": "FLNS", "FLNSC": "FLNSC", "FSNSC": "FSNSC",
                "FLNT": "FLNTOA", "FLNTC": "FLNTOAC", "FSNT": "FSNTOA", "FSNTC": "FSNTOAC",
                "SWCF": "SWCRE", "LWCF": "LWCRE", "CRE": "CRE", "CLDTOT": "CLDTOT"
           }.items()
        },

        # GPCP
        "PRECT": {"source": "GPCP", "alias": "PRECT", "fscl": 1.0, "run": gpcp_path, "template": gpcp_template, "period": "197901-201912"},
    }

    return ref_dict

In [24]:
class ErrorMetricCalculator:
    def __init__(self, regnam, tstart, tend, frequency,
                 model_list, ref_dict, exp_dict, path_in, out_path,
                 var_list=None, force=False, include_full_time=True):
        self.regnam = regnam
        self.tstart = tstart
        self.tend = tend
        self.frequency = frequency
        self.model_list = model_list
        self.ref_dict = ref_dict
        self.exp_dict = exp_dict
        self.path_in = path_in
        self.out_path = os.path.join(out_path, frequency)
        self.force = force

        self.var_dict = self.extract_var_list()
        self.var_list = var_list if var_list is not None else list(self.var_dict.keys())
        self.seasons = {
            "DJF": [12, 1, 2],
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
            "ANN": list(range(1, 13))
        }

        os.makedirs(self.out_path, exist_ok=True)
        self.start_date = np.datetime64(self.tstart)
        self.end_date = np.datetime64(self.tend)

        self.model_data_cache = {}
        self.obs_data_cache = {}

    def compute_parallel(self, metrics, n_jobs: int = 8):
        self.load_model_data()
        for var in self.var_list:
            print(f"[INFO] Processing variable: {var}")
            obs = self.read_reference_data(var)
            vinfo = self.var_dict[var]
            for exp in self.model_list:
                mod = self.read_model_data(var, vinfo, self.model_data_cache[exp])
                for metric in metrics:
                    out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_{metric}_{exp}.nc")
                    if not os.path.exists(out_file) or self.force:
                        self._compute_and_save(exp, var, vinfo, obs, mod, metric, out_file)
                    else:
                        print(f"[SKIP] {out_file} already exists. Skipping...")

    def _compute_and_save(self, exp, var, vinfo, obs, mod, metric, out_file):
        try:
            results = self._process_metric(metric, obs, mod, out_file)
            values = [results[s] for s in self.seasons]
            ds_out = xr.Dataset({
                var: xr.DataArray(values, dims="season", coords={"season": list(self.seasons)})
            })
            ds_out.to_netcdf(out_file)
        except Exception as e:
            print(f"[ERROR] Failed computing {metric} for {var}, {exp}: {e}")

    def _process_metric(self, metric, obs, mod, out_file) -> Dict[str, float]:
        weights = self.generate_coslat_weight(mod)
        results = {}
        for season, months in self.seasons.items():
            if season == "ANN":
                obs_season = obs
                mod_season = mod
            else:
                obs_season = obs.where(obs['time.month'].isin(months), drop=True)
                mod_season = mod.where(mod['time.month'].isin(months), drop=True)
            # --- diagnostics ---
            if obs_season.time.size > 0 and mod_season.time.size > 0:
                print(f"[DIAG] {season}: obs time {obs_season.time[0].values} → {obs_season.time[-1].values}, "
                      f"mod time {mod_season.time[0].values} → {mod_season.time[-1].values}, "
                      f"(n={obs_season.time.size} steps)")
            else:
                print(f"[DIAG] {season}: no data selected!")
                
            if obs_season.time.size == 0 or mod_season.time.size == 0:
                results[season] = np.nan
                continue
            results[season] = self.compute_metric(mod_season, obs_season, metric, weights)
        return results

    def read_model_data(self, var, vinfo, ds):
        vin = vinfo['alias']
        vfac = vinfo['fscl']
        ds = self.fix_coordinates(ds)
        region = self.define_region()
        ds = ds.sel(lat=slice(*region[0]), lon=slice(*region[1]))
        if var not in ds and vin not in ds:
            if var == 'PRECT':
                ds[var] = ds['PRECC'] + ds['PRECL']
            elif var == 'PRECST':
                ds[var] = ds['PRECSC'] + ds['PRECSL']
            elif var == 'CRE':
                ds[var] = ds['LWCF'] + ds['SWCF']
            else:
                raise KeyError(f"Cannot derive {var}")
        elif vin in ds:
            ds[var] = ds[vin]
            
        ds[var] = ds[var] * vfac

        return self.safe_transpose(ds[var])

    def read_reference_data(self, var):
        ref_val = self.ref_dict[var]
        ref_key = ref_val['source']
        ref_scl = ref_val['fscl']
        ref_var = ref_val['alias']
        print(f"[INFO] Loading reference data: {ref_key}")
        print(os.path.join(ref_val["run"], ref_val["template"].replace('%(year)', '*')))
        all_files = sorted(glob.glob(os.path.join(ref_val["run"], ref_val["template"].replace('%(year)', '*'))))
        start_year = int(self.tstart[:4])
        end_year = int(self.tend[:4])
        selected_files = []
        for f in all_files:
            fname = os.path.basename(f)
            try:
                year = int(fname.split('_')[-1].split('.')[0])
                if start_year <= year <= end_year:
                    selected_files.append(f)
            except ValueError:
                print(f"[WARNING] Skipping unexpected file: {fname}")
        ds_all = xr.open_mfdataset(
            selected_files,
            combine='nested',
            concat_dim='time',
            parallel=True,
            coords='minimal',
            compat='override',
            engine="netcdf4",
            decode_times=False
        )
        dates = xr.cftime_range(start=self.tstart, end=self.tend, freq="MS", calendar="365_day")
        # --- diagnostics before ---
        if 'time' in ds_all:
            print(f"[DIAG] Before: time range {ds_all.time.values[0]} → {ds_all.time.values[-1]} "
                  f"(n={ds_all.sizes['time']})")
                
        if ds_all.sizes['time'] != len(dates):
            raise ValueError(f"[ERROR] Time mismatch: {ds_all.sizes['time']} vs expected {len(dates)}")
            
        ds_all['time'] = ('time', dates)
    
        # --- diagnostics after ---
        print(f"[DIAG] After: time range {ds_all.time.values[0]} → {ds_all.time.values[-1]} "
              f"(n={ds_all.sizes['time']})")
        
        ds = self.fix_coordinates(ds_all)
        region = self.define_region()
        ds = ds.sel(lat=slice(*region[0]), lon=slice(*region[1]))
        if var not in ds and ref_var not in ds:
            if var == 'PRECT':
                ds[var] = ds['PRECC'] + ds['PRECL']
            elif var == 'PRECST':
                ds[var] = ds['PRECSC'] + ds['PRECSL']
            else:
                raise KeyError(f"Cannot derive {var}")
        elif ref_var in ds:
            ds[var] = ds[ref_var]

        ds[var] = ds[var] * ref_scl

        return self.safe_transpose(ds[var])

    def load_model_data(self):
        for exp in self.model_list:
            print(f"[INFO] Loading model data: {exp}")
            path = self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run'])
            all_files = sorted(glob.glob(os.path.join(path, f"*_h0.nc")))
            start_year = int(self.tstart[:4])
            end_year = int(self.tend[:4])
            selected_files = []
            for f in all_files:
                fname = os.path.basename(f)
                try:
                    year = int(fname.split('_')[0])
                    if start_year <= year <= end_year:
                        selected_files.append(f)
                except ValueError:
                    print(f"[WARNING] Skipping unexpected file: {fname}")
            ds_all = xr.open_mfdataset(
                selected_files,
                combine='nested',
                concat_dim='time',
                parallel=True,
                coords='minimal',
                compat='override',
                decode_times=False
            )
            dates = xr.cftime_range(start=self.tstart, end=self.tend, freq="MS", calendar="365_day")
            # --- diagnostics before ---
            if 'time' in ds_all:
                print(f"[DIAG] Before: time range {ds_all.time.values[0]} → {ds_all.time.values[-1]} "
                      f"(n={ds_all.sizes['time']})")
                
            if ds_all.sizes['time'] != len(dates):
                raise ValueError(f"[ERROR] Time mismatch: {ds_all.sizes['time']} vs expected {len(dates)}")
            
            ds_all['time'] = ('time', dates)
            # --- diagnostics after ---
            print(f"[DIAG] After: time range {ds_all.time.values[0]} → {ds_all.time.values[-1]} "
                  f"(n={ds_all.sizes['time']})")
            self.model_data_cache[exp] = ds_all
            

    def compute_metric(self, fcst, obs, metric, weights):
        obs = obs.chunk(dict(time=-1, lat=-1, lon=-1))
        fcst = fcst.chunk(dict(time=-1, lat=-1, lon=-1))

        if metric == "MEAN":
            return fcst.weighted(weights).mean(("lat", "lon", "time")).compute().item()
        elif metric == "BIAS":
            return (fcst - obs).weighted(weights).mean(("lat", "lon", "time")).compute().item()
        elif metric == "RMSE":
            return xs.rmse(obs, fcst, dim=["lat", "lon"], weights=weights, skipna=True).mean("time").compute().item()
        elif metric == "ACC":
            obs_anom = obs - obs.mean("time")
            fcst_anom = fcst - fcst.mean("time")
            acc_map = xs.pearson_r(obs_anom, fcst_anom, dim="time", skipna=True)
            return (acc_map * weights).mean(("lat", "lon")).compute().item()
        elif metric == "STD":
            return fcst.std("time").weighted(weights).mean(("lat", "lon")).compute().item()
        elif metric == "STCORR":
            corr = xs.pearson_r(obs, fcst, dim="time", skipna=True)
            return (corr * weights).mean(("lat", "lon")).compute().item()
        elif metric == "TSCORR":
            corrs = [
                xs.pearson_r(obs.isel(time=t), fcst.isel(time=t), dim=["lat", "lon"], skipna=True).compute().item()
                for t in range(fcst.time.size)
            ]
            return np.nanmean(corrs)
        else:
            raise ValueError(f"Unsupported metric: {metric}")

    def safe_transpose(self, da, dims=('time', 'lat', 'lon')):
        current_dims = set(da.dims)
        target_dims = set(dims)
        if target_dims.issubset(current_dims) and current_dims.issubset(target_dims):
            return da.transpose(*dims)
        elif target_dims.issubset(current_dims):
            da = da.squeeze()
            da = da.sel(nbnd=0) if "nbnd" in da.dims else da
            return da.transpose(*dims) if set(da.dims) >= target_dims else da
        else:
            raise ValueError(f"Cannot safely transpose {da.dims} to {dims}")

    @staticmethod
    def fix_coordinates(ds):
        if "lev" in ds.dims:
            ds = ds.isel(lev=0, drop=True)
        if ds.lon.min() > -1.0:
            ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby('lon')
        return ds

    @staticmethod
    def generate_coslat_weight(ds):
        weights = np.cos(np.deg2rad(ds.lat))
        return weights.broadcast_like(ds.isel(time=0))

    def define_region(self):
        reg_dict = {
            'global': [(-90, 90), (-180, 180)],
            'Atlantic': [(5, 55), (-95, -40)],
            'CONUS': [(25, 50), (-125, -95)],
            'Antarctic': [(-90, -50), (-180, 180)],
            'PolarN': [(50, 90), (-180, 180)],
            'Greenland': [(60, 85), (-75, -10)]
        }
        return reg_dict[self.regnam]

    @staticmethod
    def extract_var_list():
        return {
            'FLUT':    {'alias': 'FLUT',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLUTC':   {'alias': 'FLUTC',   'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLDS':    {'alias': 'FLDS',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLDSC':   {'alias': 'FLDS',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLNS':    {'alias': 'FLNS',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLNSC':   {'alias': 'FLNSC',   'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FSNS':    {'alias': 'FSNSC',   'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FSNSC':   {'alias': 'FSNS',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLNT':    {'alias': 'FLNTOA',  'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FSNT':    {'alias': 'FSNTOA',  'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FLNTC':   {'alias': 'FLNTOAC', 'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'FSNTC':   {'alias': 'FSNTOAC', 'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},
            'SWCF':    {'alias': 'SWCF',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11},            
            'LWCF':    {'alias': 'LWCF',    'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11}, 
            'CRE':     {'alias': 'CRE',     'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11}, 
            'TGCLDLWP':{'alias': 'TGCLDLWP','unit': 'kg m$^{-2}$',          'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11}, 
            'TGCLDIWP':{'alias': 'TGCLDIWP','unit': 'kg m$^{-2}$',          'fscl': 1.0,    'min': -10,  'max': 10,  'nlev': 11}, 
            'PSL':     {'alias': 'PSL',     'unit': 'hPa',                  'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'TMQ':     {'alias': 'TMQ',     'unit': 'kg m$^{-2}$',          'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'TS':      {'alias': 'TS',      'unit': 'K',                    'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'TREFHT':  {'alias': 'TREFHT',  'unit': 'K',                    'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'QFLX':    {'alias': 'QFLX',    'unit': 'mm day$^{-1}$',        'fscl': 86400.0,'min': -2,   'max': 2,   'nlev': 11},
            'SHFLX':   {'alias': 'SHFLX',   'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'LHFLX':   {'alias': 'LHFLX',   'unit': 'W m$^{-2}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'CLDTOT':  {'alias': 'CLDTOT',  'unit': '%',                    'fscl': 100.0,  'min': -2,   'max': 2,   'nlev': 11},
            'PRECT':   {'alias': 'PRECT',   'unit': 'mm day$^{-1}$',        'fscl': 8.64e7, 'min': -2,   'max': 2,   'nlev': 11,
                        'components': ["PRECC", "PRECL"], "operation": "sum", },
            'PRECST':  {'alias': 'PRECST',  'unit': 'mm day$^{-1}$',        'fscl': 8.64e7, 'min': -2,   'max': 2,   'nlev': 11,
                        'components': ["PRECSC", "PRECSL"], "operation": "sum", },
        }

In [25]:
if __name__ == "__main__":
    # Set basic paths
    top_path  = "/pscratch/sd/z/zhan391/seacrogs_scratch"
    data_path = f"{top_path}/post_data"
    out_path  = "/pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/metric_data"
    os.makedirs(out_path, exist_ok=True)
    
    # Load model experiment metadata
    exp_json = f"/pscratch/sd/z/zhan391/seacrogs_scratch/post_data/scripts/ml_exp_info.json"
    with open(exp_json, "r") as f:
        exp_dict = json.load(f)
        
    # Time and frequency
    tstart = "2012-01"
    tend   = "2016-12"
    freq   = "monthly"

    # Define reference (ERA5) metadata
    ref_dict = get_reference_data(freq = freq, data_root = data_path) 
    
    # Template for locating model files
    path_template = f"{data_path}/%(CASENAME)/{freq}"

    # Metrics and variables to compute
    metrics = ["MEAN", "BIAS", "RMSE", "ACC", "TSCORR", "STCORR"]
    variables = None  # use all default variables from extract_var_list()

    # Initialize calculator
    calculator = ErrorMetricCalculator(
        regnam="global",
        tstart=tstart,
        tend=tend,
        frequency=freq,
        model_list=list(exp_dict.keys()),
        ref_dict=ref_dict,
        exp_dict=exp_dict,
        path_in=path_template,
        out_path=out_path,
        var_list=variables,
        force=True
    )
    
    # Run calculation
    calculator.compute_parallel(metrics, n_jobs=1)

[INFO] Loading model data: CLIM
[DIAG] Before: time range 215.0 → 2009.0 (n=60)
[DIAG] After: time range 2012-01-01 00:00:00 → 2016-12-01 00:00:00 (n=60)
[INFO] Loading model data: UNet
[DIAG] Before: time range 31.0 → 1825.0 (n=60)
[DIAG] After: time range 2012-01-01 00:00:00 → 2016-12-01 00:00:00 (n=60)
[INFO] Loading model data: UNetMP
[DIAG] Before: time range 31.0 → 1825.0 (n=60)
[DIAG] After: time range 2012-01-01 00:00:00 → 2016-12-01 00:00:00 (n=60)
[INFO] Loading model data: IUNet
[DIAG] Before: time range 31.0 → 1825.0 (n=60)
[DIAG] After: time range 2012-01-01 00:00:00 → 2016-12-01 00:00:00 (n=60)
[INFO] Loading model data: MnM
[DIAG] Before: time range 31.0 → 1825.0 (n=60)
[DIAG] After: time range 2012-01-01 00:00:00 → 2016-12-01 00:00:00 (n=60)
[INFO] Processing variable: FLUT
[INFO] Loading reference data: CERES
/pscratch/sd/z/zhan391/seacrogs_scratch/post_data/CERES-OAFlux/monthly/CERES-OAFlux_*.nc
[DIAG] Before: time range 4337 → 6133 (n=60)
[DIAG] After: time range 201